# NHAMCS 2022 ED Admission Prediction — Exploratory Data Analysis

Milestone 2/3 (Sprint 1: Machine Learning Data Pipeline). Presentation layer over the reusable, tested modules in `ML/eda/` — every table and chart below is computed by importing and calling that code directly, not reimplemented here.

Full markdown report: `ML/reports/eda/eda_report.md`. Raw CSV outputs: `ML/reports/eda/`.

Run on the **raw** dataset (before cleaning), so this reflects data quality *before* any fixes — see `ML/reports/cleaning/cleaning_report.md` for what was done about it.

In [1]:
# Make the repo root importable regardless of the notebook's working
# directory (nbconvert defaults to this notebook's own folder).
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import Image, display

from ML.ingestion.config import DEFAULT_CONFIG_PATH, load_config
from ML.ingestion.loader import load_dataset
from ML.eda.type_classification import split_columns_by_type
from ML.eda.missing_values import build_missing_value_report, summarize_missing_values
from ML.eda.duplicates import analyze_duplicates
from ML.eda.numerical_summary import build_numerical_summary
from ML.eda.categorical_summary import build_categorical_summary
from ML.eda.class_imbalance import compute_derived_target, analyze_class_imbalance
from ML.eda.correlation import build_correlation_matrix, top_correlated_pairs, target_correlation

FIGURES_DIR = "../reports/eda/figures"

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 20)

## Load Dataset

In [2]:
config = load_config(DEFAULT_CONFIG_PATH)
dataframe, metadata = load_dataset(config)
print(f"Shape: {dataframe.shape[0]} rows x {dataframe.shape[1]} columns")
print(f"Encoding: {metadata.file_encoding}")

Shape: 16025 rows x 913 columns
Encoding: WINDOWS-1252


## 1. Missing Value Analysis

NHAMCS also encodes missingness via negative sentinel codes (-7/-8/-9), handled during cleaning (Milestone 3) — the numbers below reflect only native/true NaN.

In [3]:
missing_report = build_missing_value_report(dataframe)
missing_summary = summarize_missing_values(missing_report, dataframe.size)
missing_summary

{'total_cells': 14630825,
 'total_missing_cells': 1325889,
 'overall_missing_percentage': 9.0623,
 'columns_with_no_missing_values': 822,
 'columns_over_50_percent_missing': 85,
 'fully_missing_columns': []}

In [4]:
missing_report.head(15)

,variable_name,missing_count,missing_percentage
0,PRESCR30,16024,99.99
1,COMSTAT30,16024,99.99
2,CONTSUB30,16024,99.99
3,COMSTAT29,16020,99.97
4,CONTSUB29,16020,99.97
5,PRESCR29,16020,99.97
6,CONTSUB28,16013,99.93
7,COMSTAT28,16013,99.93
8,PRESCR28,16013,99.93
9,CONTSUB27,16011,99.91


![Missing value heatmap](../reports/eda/figures/missing_value_heatmap.png)

## 2. Duplicate Analysis

In [5]:
analyze_duplicates(dataframe)

{'duplicate_row_count': 0,
 'duplicate_row_percentage': 0.0,
 'duplicate_row_indices': []}

## 3. Numerical Summaries

Numerical vs. categorical is a cardinality-based heuristic (>20 unique values ⇒ numerical) — see `ML/eda/type_classification.py` docstring for the caveat. `negative_value_count` flags NHAMCS sentinel codes contaminating raw statistics.

In [6]:
numerical_columns, categorical_columns = split_columns_by_type(dataframe)
print(f"{len(numerical_columns)} numerical, {len(categorical_columns)} categorical")

numerical_summary = build_numerical_summary(dataframe, numerical_columns)
numerical_summary.sort_values("negative_value_percentage", ascending=False)

37 numerical, 876 categorical


,variable_name,count,mean,std,min,p25,median,p75,max,negative_value_count,negative_value_percentage,iqr_lower_bound,iqr_upper_bound,outlier_count,outlier_percentage
30,OBSSTAY,16025,12.333,207.323,-9.00000,-7.000000,-7.000000,-7.00000,7397.00000,15700,97.97,-7.000,-7.000,391,2.44
3,AGEDAYS,16025,-1.902,35.444,-7.00000,-7.000000,-7.000000,-7.00000,364.00000,15597,97.33,-7.000,-7.000,428,2.67
36,BOARDED,16025,15.042,167.678,-9.00000,-7.000000,-7.000000,-7.00000,4206.00000,14352,89.56,-7.000,-7.000,2116,13.20
29,LOS,16025,-5.378,5.080,-9.00000,-7.000000,-7.000000,-7.00000,99.00000,14002,87.38,-7.000,-7.000,2121,13.24
14,RFV5,16025,3122.033,9973.227,-9.00000,-9.000000,-9.000000,-9.00000,89990.00000,13919,86.86,-9.000,-9.000,2106,13.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,HOSPCODE,16025,92.747,56.447,1.00000,41.000000,90.000000,145.00000,188.00000,0,0.00,-115.000,301.000,0,0.00
32,PATCODE,16025,49.729,32.603,1.00000,23.000000,46.000000,73.00000,173.00000,0,0.00,-52.000,148.000,89,0.56
33,CPSUM,16025,51466.690,50022.714,1.00000,14.000000,100002.000000,100108.00000,100185.00000,0,0.00,-150127.000,250249.000,0,0.00
34,PATWT,16025,9697.207,9713.634,89.85488,3166.616040,6523.824650,11760.94339,57925.50019,0,0.00,-9724.875,24652.434,1386,8.65


### Distributions (curated key variables)

![age_histogram.png](../reports/eda/figures/age_histogram.png)
![waittime_histogram.png](../reports/eda/figures/waittime_histogram.png)
![lov_histogram.png](../reports/eda/figures/lov_histogram.png)
![tempf_histogram.png](../reports/eda/figures/tempf_histogram.png)
![pulse_histogram.png](../reports/eda/figures/pulse_histogram.png)
![respr_histogram.png](../reports/eda/figures/respr_histogram.png)
![bpsys_histogram.png](../reports/eda/figures/bpsys_histogram.png)
![bpdias_histogram.png](../reports/eda/figures/bpdias_histogram.png)
![popct_histogram.png](../reports/eda/figures/popct_histogram.png)
![boarded_histogram.png](../reports/eda/figures/boarded_histogram.png)

## 4. Categorical Summaries

Top 5 categories, cardinality, and rare-category count (<1% frequency) per variable.

In [7]:
categorical_summary = build_categorical_summary(dataframe, categorical_columns)
categorical_summary.sort_values("n_unique", ascending=False).head(20)

,variable_name,n_unique,missing_count,missing_percentage,top_values,rare_category_count
35,DIAG1,1549,0,0.0,U071=508 (3.2%); R079=406 (2.5%); R109=367 (2....,1534
2,ARRTIME,1438,0,0.0,-9=243 (1.5%); 1124=29 (0.2%); 1440=29 (0.2%);...,1437
36,DIAG2,1308,0,0.0,-9=7635 (47.6%); I10-=199 (1.2%); Z208=171 (1....,1304
37,DIAG3,989,0,0.0,-9=11774 (73.5%); I10-=158 (1.0%); Z208=121 (0...,988
130,MED1,906,0,0.0,-9=3634 (22.7%); 28495=931 (5.8%); 32905=846 (...,891
131,MED2,892,0,0.0,-9=6885 (43.0%); 28495=502 (3.1%); 92105=468 (...,884
132,MED3,828,0,0.0,-9=9709 (60.6%); 28495=324 (2.0%); 92105=292 (...,824
133,MED4,737,0,0.0,-9=11771 (73.5%); 28495=210 (1.3%); 92105=149 ...,735
134,MED5,675,0,0.0,-9=13149 (82.1%); 28495=102 (0.6%); 92105=95 (...,674
38,DIAG4,668,0,0.0,-9=13743 (85.8%); I10-=91 (0.6%); Z208=61 (0.4...,667


### Distributions (curated key variables)

![sex_bar.png](../reports/eda/figures/sex_bar.png)
![ager_bar.png](../reports/eda/figures/ager_bar.png)
![racer_bar.png](../reports/eda/figures/racer_bar.png)
![ethun_bar.png](../reports/eda/figures/ethun_bar.png)
![arrems_bar.png](../reports/eda/figures/arrems_bar.png)
![paytyper_bar.png](../reports/eda/figures/paytyper_bar.png)
![immedr_bar.png](../reports/eda/figures/immedr_bar.png)
![painscale_bar.png](../reports/eda/figures/painscale_bar.png)
![vdayr_bar.png](../reports/eda/figures/vdayr_bar.png)
![vmonth_bar.png](../reports/eda/figures/vmonth_bar.png)
![stay24_bar.png](../reports/eda/figures/stay24_bar.png)

## 5. Class Imbalance Analysis

Target: `hospital_admission`, derived as `ADMITHOS == 1 or OBSHOS == 1` (see `ML/reports/target_and_survey_variables.md`).

In [8]:
target = compute_derived_target(dataframe, config)
analyze_class_imbalance(target)

{'counts': {'0': 13904, '1': 2121},
 'percentages': {'0': 86.76, '1': 13.24},
 'imbalance_ratio': 6.56}

![Target class distribution](../reports/eda/figures/target_class_distribution_bar.png)

## 6. Correlation Analysis

In [9]:
correlation_matrix = build_correlation_matrix(dataframe, numerical_columns)
top_correlated_pairs(correlation_matrix)

,variable_a,variable_b,correlation
0,RFV1,RFV13D,1.000
1,RFV3,RFV33D,1.000
2,RFV2,RFV23D,1.000
3,RFV4,RFV43D,1.000
4,RFV5,RFV53D,1.000
5,BPSYSD,BPDIASD,0.982
6,BPSYS,BPDIAS,0.886
7,PULSED,RESPRD,0.840
8,PULSED,BPDIASD,0.771
9,RESPRD,BPSYSD,0.767


In [10]:
target_correlation(dataframe, numerical_columns, target)

,variable_name,correlation_with_target
0,LOS,0.817393
1,TOTDIAG,0.393930
2,NUMGIV,0.364951
3,BOARDED,0.336584
4,NUMMED,0.271110
5,AGE,0.256138
6,LOV,0.241855
7,EDWT,-0.173051
8,BPSYSD,0.166563
9,BPDIASD,0.156798


![Correlation heatmap](../reports/eda/figures/correlation_heatmap.png)

## 7. Key Findings

- **`LOS` (length of hospital stay) is a leakage risk, not a usable predictor.** It correlates ~0.82 with the target, but it's only populated for already-admitted visits and unknown at prediction time — excluded from the model-ready feature set (see `ML/feature_engineering/leakage_exclusion.py`).
- **`RFV1-5` are near-duplicates of `RFV*3D`** (correlation 1.0) — the detailed 5-digit reason-for-visit codes were dropped in favor of the coarser 3-digit recode during feature engineering (fewer categories, more interpretable one-hot encoding).
- Class imbalance is moderate (~13% admitted, 6.6:1 ratio) — not severe enough to require resampling, but worth stratifying splits on (done in Milestone 7).

## Next Steps

- Cleaning: `ML/reports/cleaning/cleaning_report.md`
- Feature engineering: `ML/reports/feature_engineering/feature_engineering_report.md`
- Feature selection: `ML/reports/feature_engineering/feature_selection_report.md`
- Full pipeline: `ML/pipeline/preprocessing_pipeline.py`